# 🧠 Notebook 04: Generation & Prompt Engineering

Trong notebook này, chúng ta sẽ khảo sát và tinh chỉnh tầng sinh câu trả lời của LLM:
1. **Prompt Builder**: Khám phá kỹ thuật cấu trúc prompt chống ảo giác (`qa_prompt.yaml`)
2. **Groq LLM Client**: Kiểm thử mô hình tốc độ cao `openai/gpt-oss-120b` trên Groq Cloud
3. **Streaming Generation**: Minh họa sinh câu trả lời theo thời gian thực (real-time tokens)
4. **Điều chỉnh Siêu tham số (Hyperparameters)**: Đánh giá tác động của `temperature` trong RAG
5. **Kiểm thử Từ chối Trả lời**: Kiểm tra phản ứng của LLM khi ngữ cảnh không đủ thông tin

## 1. Thiết lập Môi trường & Imports

In [ ]:
import sys
import time
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import get_config
from src.generation.prompt_builder import PromptBuilder
from src.generation.groq_llm import GroqLLM
from src.generation.generator import Generator, get_llm
from src.schemas.response import RetrievedChunk

cfg = get_config()
print(f"✅ Đã nạp thành công module Generation. Provider: {cfg.llm.default_provider} ({cfg.llm.groq.model_name})")

## 2. Khám phá Prompt Builder & Cấu trúc Prompt

`PromptBuilder` tự động đọc template từ `configs/prompts/qa_prompt.yaml` và ghép nối các đoạn chunks được trích xuất.

In [ ]:
builder = PromptBuilder()

# Giả lập 2 chunks kết quả tìm kiếm
mock_chunks = [
    RetrievedChunk(
        chunk_id="chunk-001",
        text="Mô hình DenseNet-201 đạt độ chính xác 93% trên tập dữ liệu kiểm thử phân loại 5 loại bệnh về răng miệng.",
        score=0.92,
        rank=1,
        metadata={"file_name": "paper_Phân tích và xử lí ảnh.docx", "section_title": "Kết luận", "page_number": 18},
    ),
    RetrievedChunk(
        chunk_id="chunk-002",
        text="Ảnh X-quang đầu vào có kích thước 224x224x3, đi qua Conv Stem 7x7 và MaxPool 3x3 trước khi vào các khối Dense Block.",
        score=0.88,
        rank=2,
        metadata={"file_name": "paper_Phân tích và xử lí ảnh.docx", "section_title": "Kiến trúc mạng DenseNet201"},
    ),
]

sys_prompt, user_prompt = builder.build_prompt(
    query="Mô hình đạt độ chính xác bao nhiêu và đầu vào là ảnh kích thước gì?",
    chunks=mock_chunks,
)

print("=== SYSTEM PROMPT ===")
print(sys_prompt[:300] + "...\n")
print("=== USER PROMPT HOÀN CHỈNH ===")
print(user_prompt)

## 3. Sinh câu trả lời với Groq LLM

Gửi prompt hoàn chỉnh tới Groq LLM API và đo lường thời gian phản hồi.

In [ ]:
generator = Generator()

t0 = time.time()
answer = generator.generate_answer(
    query="Mô hình đạt độ chính xác bao nhiêu và đầu vào là ảnh kích thước gì?",
    chunks=mock_chunks,
)
latency = time.time() - t0

print(f"⏱️ Thời gian LLM phản hồi: {latency:.2f}s\n")
display(Markdown("### 🤖 Phản hồi của LLM:\n\n" + answer))

## 4. Thử nghiệm Streaming Token Generation

Nhận từng token qua Generator iterator để mô phỏng trải nghiệm gõ chữ real-time.

In [ ]:
stream_query = "Tóm tắt ngắn gọn kiến trúc mạng và kết quả thực nghiệm."

print(f"❓ Câu hỏi: {stream_query}\n")
print("🤖 AI đang gõ từng chữ:\n")

stream_iter = generator.stream_answer(query=stream_query, chunks=mock_chunks)
for token in stream_iter:
    print(token, end="", flush=True)

print("\n\n✅ Đã hoàn tất streaming.")

## 5. Khảo sát Siêu tham số: `temperature = 0.0` vs `temperature = 0.7`

Trong bài toán RAG truy xuất tài liệu, `temperature` thấp (0.0 - 0.2) thường được ưu tiên để đảm bảo tính tất định và trung thành với ngữ cảnh.

In [ ]:
print("🔹 1. VỚI TEMPERATURE = 0.0 (Tất định, tập trung chính xác vào ngữ cảnh):")
ans_temp_0 = generator.generate_answer(
    query="Mô hình có độ chính xác là bao nhiêu?",
    chunks=mock_chunks,
    temperature=0.0,
)
print(ans_temp_0 + "\n")

print("🔹 2. VỚI TEMPERATURE = 0.7 (Văn phong linh hoạt hơn):")
ans_temp_7 = generator.generate_answer(
    query="Mô hình có độ chính xác là bao nhiêu?",
    chunks=mock_chunks,
    temperature=0.7,
)
print(ans_temp_7)

## 6. Kiểm thử Từ chối Trả lời khi Ngữ cảnh Không Đủ Thông tin

Kiểm tra xem khi cung cấp ngữ cảnh về nha khoa nhưng người dùng hỏi về du lịch, AI có từ chối theo nguyên tắc chống ảo giác hay không.

In [ ]:
unsupported_query = "Nhiệt độ sôi của nước ở độ cao 2000m là bao nhiêu?"

answer_rejected = generator.generate_answer(
    query=unsupported_query,
    chunks=mock_chunks,
)

display(Markdown("### 🤖 Kết quả xử lý câu hỏi ngoài phạm vi:\n\n" + answer_rejected))